# Laboratorio Colab · De la evidencia a los modelos

**Casos:** NovaRetail + InsureCO  
**Audiencia:** MBA — no se requiere experiencia previa en programación  
**Propósito:** reconocer qué pregunta responde cada familia, ejecutar un pipeline reproducible, interpretar métricas y detectar cuándo un modelo **no** justifica una decisión.

Este notebook adapta materiales de regresión, clasificación, árboles, Random Forest, clustering, PCA y anomalías de las Partes 2–4. No presenta los modelos como recetas. En cada laboratorio seguimos:

> **pregunta → línea base → modelo → evaluación fuera de muestra → errores → decisión → límite**

### Cómo trabajar

1. Ejecuta las celdas en orden; no saltes la carga y la auditoría.
2. En la sección 2 aparecerá un botón **Elegir archivos** de Colab.
3. La ruta recomendada es cargar un solo archivo: `datos_semana2.zip`.
4. También puedes seleccionar juntos `ventas.csv`, `clientes.csv`, `productos.csv` e `insureco.csv`.
5. Lee la predicción o pregunta **antes** de ejecutar cada resultado.
6. No necesitas memorizar importaciones: debes explicar qué entra, qué sale, cómo se evalúa y qué límite existe.


## 0. Objetivos y mapa

| Laboratorio | Pregunta empresarial | Familia | Salida |
|---|---|---|---|
| 1 | ¿Qué variables se mueven juntas? | correlación | coeficiente/gráfica |
| 2 | ¿Qué costo de póliza podemos estimar? | regresión | número |
| 3 | ¿Qué transacción podría cancelarse? | clasificación | probabilidad/clase |
| 4 | ¿Qué reglas encuentra un árbol? | árbol y Random Forest | reglas/clase |
| 5 | ¿Qué perfiles de clientes aparecen? | K-means | cluster |
| 6 | ¿Podemos resumir varias métricas? | PCA | componentes |
| 7 | ¿Qué transacciones son inusuales? | Isolation Forest | alerta/score |

**Importante:** encontrar asociación, importancia o un cluster no prueba causalidad.

### Ruta presencial sugerida · seis horas efectivas

| Tiempo | Trabajo |
|---:|---|
| 35 min | conceptos, preguntas y tipos de aprendizaje |
| 35 min | carga, unidad de análisis, llaves y tabla analítica |
| 35 min | correlación y lenguaje prudente |
| 75 min | regresión simple/múltiple y evaluación |
| 100 min | clasificación, umbrales, árbol y Random Forest |
| 65 min | clustering y evaluación de perfiles |
| 15 min | PCA, anomalías y cierre |

El notebook contiene extensiones suficientes para detenerse, modificar parámetros y discutir; no está diseñado para ser ejecutado pasivamente de principio a fin.


## 0.1 Glosario mínimo antes de empezar

| Término | Significado práctico |
|---|---|
| **Observación** | una fila o caso de la unidad de análisis |
| **Variable** | una característica medida en cada observación |
| **Objetivo `y`** | resultado que un modelo supervisado intenta estimar |
| **Predictores `X`** | información disponible para construir la estimación |
| **Parámetro** | valor aprendido por el modelo, como un coeficiente |
| **Hiperparámetro** | decisión configurada antes de entrenar, como profundidad o número de clusters |
| **Entrenar** | aprender patrones utilizando un conjunto de observaciones |
| **Generalizar** | funcionar razonablemente con observaciones nuevas |
| **Línea base** | regla simple que un modelo debe superar |
| **Pipeline** | secuencia reproducible de preparación y modelado |

### Tres niveles que no deben mezclarse

1. **Describir:** “las personas fumadoras tienen costos observados diferentes”.
2. **Predecir:** “estas variables permiten estimar parte del costo fuera de muestra”.
3. **Explicar causalmente:** “fumar produjo exactamente este cambio”.

Los dos primeros pueden estudiarse aquí. El tercero exige un diseño causal que este notebook no proporciona.


## 0.2 Supervisado vs. no supervisado

**Supervisado:** existe una respuesta histórica conocida. El modelo compara sus predicciones con esa respuesta.

```text
X: edad + IMC + fumador + región  ──►  modelo  ──►  y: costo
```

**No supervisado:** no existe una etiqueta correcta. Buscamos estructura y después evaluamos si es estable, interpretable y útil.

```text
métricas de clientes  ──►  algoritmo  ──►  grupos numerados
                                           ↓
                              interpretación del analista
```

**Pregunta de control:** si no puedes decir si existe una `y` conocida, todavía no has definido el tipo de tarea.


## 1. Preparar el entorno

Las librerías utilizadas ya están instaladas en Google Colab. Fijamos una semilla para que las particiones y resultados aleatorios puedan reproducirse.


In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, silhouette_score
)

SEMILLA = 42
np.random.seed(SEMILLA)
warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

print("Entorno listo. Semilla:", SEMILLA)


## 2. Cargar los datos con el selector de Colab

### Archivos esperados

**Opción recomendada:** carga `datos_semana2.zip`; contiene los cuatro CSV y evita seleccionar uno por uno.

**Opción alternativa:** selecciona simultáneamente:

- `ventas.csv`
- `clientes.csv`
- `productos.csv`
- `insureco.csv`

Al ejecutar la siguiente celda, Colab mostrará el botón **Elegir archivos**. También puedes arrastrar el ZIP o los cuatro CSV al selector. Espera a que termine la carga antes de continuar.

En ejecución local, la misma celda busca una carpeta `data` cercana. No subas datos sensibles o identificables a servicios externos sin autorización.

> **Si aparece un error de nombres:** no renombres arbitrariamente los archivos; confirma la lista anterior y vuelve a ejecutar la celda.


In [ ]:
import zipfile

ARCHIVOS = ["ventas.csv", "clientes.csv", "productos.csv", "insureco.csv"]
NOMBRE_ZIP = "datos_semana2.zip"

candidatos = [
    Path("/content"),
    Path.cwd(),
    Path.cwd() / "data",
    Path.cwd().parent / "data",
    Path.cwd().parent.parent / "data",
]

data_dir = next(
    (ruta for ruta in candidatos
     if all((ruta / nombre).exists() for nombre in ARCHIVOS)),
    None
)

if data_dir is None and "google.colab" in sys.modules:
    from google.colab import files
    print("Selecciona datos_semana2.zip o los cuatro CSV indicados arriba.")
    print("Puedes arrastrarlos al selector o usar el botón Elegir archivos.")
    cargados = files.upload()
    data_dir = Path("/content")

    if NOMBRE_ZIP in cargados:
        with zipfile.ZipFile(data_dir / NOMBRE_ZIP) as paquete:
            paquete.extractall(data_dir)
        print("ZIP extraído correctamente.")

if data_dir is None:
    raise FileNotFoundError(
        "No encuentro los datos. Carga datos_semana2.zip o los cuatro CSV, "
        "o ubícalos juntos en la carpeta actual/data/."
    )

faltan = [nombre for nombre in ARCHIVOS if not (data_dir / nombre).exists()]
if faltan:
    raise FileNotFoundError(
        "Faltan estos archivos: " + ", ".join(faltan) +
        ". Vuelve a ejecutar la celda y selecciónalos."
    )

ventas_raw = pd.read_csv(data_dir / "ventas.csv")
clientes = pd.read_csv(data_dir / "clientes.csv")
productos = pd.read_csv(data_dir / "productos.csv")
insureco = pd.read_csv(data_dir / "insureco.csv")

ESQUEMAS = {
    "ventas.csv": (
        ventas_raw,
        {"id_venta", "fecha", "id_cliente", "id_producto", "canal",
         "cantidad", "valor_venta", "descuento", "estado"},
    ),
    "clientes.csv": (
        clientes,
        {"id_cliente", "segmento", "ciudad", "region", "edad", "sexo",
         "fecha_registro"},
    ),
    "productos.csv": (
        productos,
        {"id_producto", "categoria", "precio_lista", "costo_unitario"},
    ),
    "insureco.csv": (
        insureco,
        {"age", "sex", "bmi", "children", "smoker", "region", "charges"},
    ),
}

errores_esquema = []
for nombre, (tabla, requeridas) in ESQUEMAS.items():
    faltantes = sorted(requeridas - set(tabla.columns))
    if faltantes:
        errores_esquema.append(f"{nombre}: faltan {faltantes}")

if errores_esquema:
    raise ValueError(
        "Los nombres de columnas no corresponden a los datos del curso:\n- "
        + "\n- ".join(errores_esquema)
    )

dimensiones_referencia = {
    "ventas.csv": (501, 11),
    "clientes.csv": (180, 7),
    "productos.csv": (40, 6),
    "insureco.csv": (1338, 7),
}
dimensiones_actuales = {
    "ventas.csv": ventas_raw.shape,
    "clientes.csv": clientes.shape,
    "productos.csv": productos.shape,
    "insureco.csv": insureco.shape,
}

print("Archivos y columnas obligatorias: validados.")
for nombre, esperada in dimensiones_referencia.items():
    actual = dimensiones_actuales[nombre]
    estado = "OK" if actual == esperada else f"REVISAR (esperada {esperada})"
    print(f"  {nombre}: {actual} — {estado}")

print("Carpeta:", data_dir.resolve())
print("ventas:", ventas_raw.shape,
      "| clientes:", clientes.shape,
      "| productos:", productos.shape,
      "| insureco:", insureco.shape)
display(ventas_raw.head(3))


### Checkpoint de carga

La salida correcta debe mostrar aproximadamente:

```text
ventas:    501 filas × 11 columnas
clientes:  180 filas × 7 columnas
productos:  40 filas × 6 columnas
insureco: 1.338 filas × 7 columnas
```

Si una dimensión es distinta, detente. Un archivo incorrecto puede permitir que el código avance y aun así invalidar todo el análisis.


## 3. Auditoría antes de modelar

La unidad de análisis de `ventas` es una transacción. Antes de hacer un join revisamos llaves, faltantes, clases y duplicados. Un modelo puede ejecutar perfectamente sobre una tabla conceptualmente equivocada.


In [ ]:
auditoria = pd.DataFrame({
    "tabla": ["ventas", "clientes", "productos"],
    "filas": [len(ventas_raw), len(clientes), len(productos)],
    "columnas": [ventas_raw.shape[1], clientes.shape[1], productos.shape[1]],
    "faltantes": [ventas_raw.isna().sum().sum(),
                  clientes.isna().sum().sum(),
                  productos.isna().sum().sum()],
    "llave_unica": [ventas_raw["id_venta"].is_unique,
                    clientes["id_cliente"].is_unique,
                    productos["id_producto"].is_unique],
})
display(auditoria)

duplicadas = ventas_raw[ventas_raw.duplicated("id_venta", keep=False)]
print("Filas con id_venta repetido:", len(duplicadas))
display(duplicadas)


NovaRetail contiene una transacción duplicada exactamente. Conservamos `raw`, creamos una copia limpia y validamos el efecto. Esta decisión sería distinta si las filas repetidas representaran eventos legítimos.


In [ ]:
ventas = ventas_raw.drop_duplicates(subset="id_venta", keep="first").copy()
ventas["fecha"] = pd.to_datetime(ventas["fecha"], errors="coerce")

control = pd.DataFrame({
    "momento": ["raw", "sin duplicado exacto"],
    "filas": [len(ventas_raw), len(ventas)],
    "ids_unicos": [ventas_raw["id_venta"].nunique(), ventas["id_venta"].nunique()],
    "valor_total": [ventas_raw["valor_venta"].sum(), ventas["valor_venta"].sum()],
})
display(control)

assert ventas["id_venta"].is_unique
assert clientes["id_cliente"].is_unique
assert productos["id_producto"].is_unique


### Construir una tabla analítica

Agregamos edad/segmento del cliente y precio/costo del producto. Usamos `validate="many_to_one"` para exigir que muchas ventas correspondan a una sola fila de cada dimensión.


In [ ]:
clientes_modelo = clientes[["id_cliente", "segmento", "edad", "sexo"]]
productos_modelo = productos[["id_producto", "precio_lista", "costo_unitario", "subcategoria"]]

modelo = (
    ventas
    .merge(clientes_modelo, on="id_cliente", how="left", validate="many_to_one")
    .merge(productos_modelo, on="id_producto", how="left", validate="many_to_one")
)

print("Filas antes/después del join:", len(ventas), len(modelo))
print("Faltantes incorporados por el join:")
display(modelo[["segmento", "edad", "precio_lista", "costo_unitario"]].isna().sum())
assert len(modelo) == len(ventas)


## Laboratorio 1 · Correlación

**Pregunta:** ¿qué variables numéricas se mueven juntas?

La correlación es una puerta de entrada, no una conclusión causal. Primero miramos la forma de la nube; después usamos un coeficiente para resumirla.

### Intuición técnica

Un coeficiente de correlación está entre −1 y 1:

- cerca de **1**: cuando una variable aumenta, la otra tiende a aumentar;
- cerca de **−1**: cuando una aumenta, la otra tiende a disminuir;
- cerca de **0**: no se observa una asociación fuerte del tipo que mide el coeficiente.

La magnitud no tiene una interpretación empresarial universal. `0,30` puede ser relevante en un contexto y trivial en otro.

| Coeficiente | Qué resume | Cuándo puede fallar |
|---|---|---|
| Pearson | cercanía a una relación lineal | extremos o relaciones curvas |
| Spearman | orden/rangos y relación monotónica | relaciones no monotónicas |

### Cuatro precauciones

1. Correlación no implica causalidad.
2. Un valor cercano a cero no descarta relaciones curvas.
3. Una tercera variable puede explicar parte de la asociación.
4. Mezclar grupos puede crear u ocultar una correlación.


In [ ]:
# Ejemplo mínimo: misma tendencia, distinta forma.
ejemplo = pd.DataFrame({
    "x": np.arange(1, 11),
    "lineal": np.arange(1, 11) * 2 + [0, 1, -1, 0, 1, -1, 0, 1, -1, 0],
    "curva": (np.arange(1, 11) - 5.5) ** 2,
})

display(ejemplo.corr().round(2))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.scatterplot(data=ejemplo, x="x", y="lineal", ax=axes[0])
axes[0].set_title("Relación aproximadamente lineal")
sns.scatterplot(data=ejemplo, x="x", y="curva", ax=axes[1])
axes[1].set_title("Relación clara, pero no lineal")
plt.tight_layout()
plt.show()


**Antes de seguir:** en la gráfica curva existe un patrón visible aunque la correlación lineal sea cercana a cero. Por eso nunca se interpreta `cor()` sin inspeccionar los datos.


In [ ]:
numericas = ["cantidad", "descuento", "valor_venta", "edad", "precio_lista", "costo_unitario"]
correlaciones = modelo[numericas].corr(method="pearson")
display(correlaciones.round(2))

plt.figure(figsize=(8, 5))
sns.heatmap(correlaciones, annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Correlaciones de Pearson")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=modelo, x="precio_lista", y="valor_venta",
                hue="cantidad", alpha=.65, ax=axes[0])
axes[0].set_title("Precio de lista y valor de venta")

sns.scatterplot(data=modelo, x="descuento", y="valor_venta",
                hue="canal", alpha=.65, ax=axes[1])
axes[1].set_title("Descuento y valor de venta")
plt.tight_layout()
plt.show()


**Discusión**

1. ¿Una correlación alta entre precio de lista y valor de venta era esperable?
2. ¿Una asociación entre descuento y valor demuestra que el descuento causó ventas?
3. ¿Qué variable podría cambiar la lectura de esa relación?

### Microactividad · 10 minutos

Sustituye `pearson` por `spearman` y compara. Después calcula por separado dentro de cada canal:

```python
modelo.groupby("canal")[["descuento", "valor_venta"]].corr()
```

Escribe una frase con el patrón agregado y otra con los segmentos. Si cambian, explica por qué una cifra global era insuficiente.


## Laboratorio 2 · Regresión con InsureCO

**Pregunta:** ¿podemos estimar `charges` para una persona asegurada nueva?

Compararemos:

- una **línea base** que siempre predice el promedio;
- regresión simple con `bmi`;
- regresión múltiple con edad, IMC, hijos, sexo, hábito de fumar y región.

### Qué hace una regresión

Una regresión construye una regla para producir un número. En el caso lineal simple:

```text
costo estimado = intercepto + pendiente × IMC
```

- **intercepto:** punto de partida matemático del modelo;
- **pendiente:** cambio promedio estimado de la salida cuando cambia el predictor;
- **residuo:** diferencia entre el valor observado y la predicción.

En la regresión múltiple agregamos predictores. El coeficiente de una variable se interpreta **manteniendo constantes las demás incluidas**, pero sigue representando asociación dentro del modelo, no causalidad.

### Datos InsureCO

| Variable | Papel |
|---|---|
| `charges` | objetivo numérico `y` |
| `age`, `bmi`, `children` | predictores numéricos |
| `sex`, `smoker`, `region` | predictores categóricos |

Separamos entrenamiento y prueba antes de ajustar el preprocesamiento. Este es un ejercicio predictivo: un coeficiente o buen desempeño no convierte una variable en causa del costo.


### Paso 1 · Explorar antes de modelar

**Predicción docente:** compara media y mediana de `charges`. Si son distintas, ¿qué esperas ver en el histograma? ¿Qué grupo podría explicar parte de la asimetría?


In [ ]:
display(insureco.describe(include="all").T)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(data=insureco, x="charges", bins=30, ax=axes[0])
axes[0].set_title("Distribución de charges")
sns.scatterplot(data=insureco, x="bmi", y="charges", alpha=.45, ax=axes[1])
axes[1].set_title("IMC y costo")
sns.boxplot(data=insureco, x="smoker", y="charges", ax=axes[2])
axes[2].set_title("Costo por hábito de fumar")
plt.tight_layout()
plt.show()


**Lectura esperada:** `charges` presenta una cola hacia valores altos. El IMC por sí solo muestra mucha dispersión; la condición de fumador separa grupos importantes. Esto anticipa que una regresión simple puede quedarse corta y una múltiple puede mejorar.

Esta lectura es exploratoria. Todavía no hemos medido capacidad predictiva fuera de muestra.


In [ ]:
objetivo_reg = "charges"
variables_reg = ["age", "bmi", "children", "sex", "smoker", "region"]

X_reg = insureco[variables_reg]
y_reg = insureco[objetivo_reg]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=.25, random_state=SEMILLA
)

print("Entrenamiento:", Xr_train.shape, "| Prueba:", Xr_test.shape)


### Paso 2 · Entrenamiento, prueba y línea base

Imagina un examen:

- **entrenamiento:** ejemplos usados para estudiar;
- **prueba:** preguntas reservadas que el modelo no vio;
- **generalización:** desempeño en esa prueba;
- **sobreajuste:** aprender detalles del entrenamiento que no se repiten.

La línea base predice siempre el promedio observado en entrenamiento. No es inteligente, pero establece el mínimo que una solución más compleja debería superar.


In [ ]:
def metricas_regresion(nombre, real, pred):
    return {
        "modelo": nombre,
        "MAE": mean_absolute_error(real, pred),
        "RMSE": mean_squared_error(real, pred) ** .5,
        "R2": r2_score(real, pred),
    }

resultados_reg = []

base_reg = DummyRegressor(strategy="mean")
base_reg.fit(Xr_train[["bmi"]], yr_train)
pred_base = base_reg.predict(Xr_test[["bmi"]])
resultados_reg.append(metricas_regresion("Línea base: media", yr_test, pred_base))

simple = LinearRegression()
simple.fit(Xr_train[["bmi"]], yr_train)
pred_simple = simple.predict(Xr_test[["bmi"]])
resultados_reg.append(metricas_regresion("Lineal simple", yr_test, pred_simple))

pd.DataFrame(resultados_reg).set_index("modelo").round(2)


### Cómo leer las métricas de regresión

| Métrica | Lectura | Mejor dirección |
|---|---|---|
| MAE | error absoluto promedio en unidades de `charges` | menor |
| RMSE | error que penaliza especialmente fallos grandes | menor |
| R² | variación explicada frente a predecir el promedio | mayor; puede ser negativo |

Un R² negativo indica que el modelo funciona peor que la referencia del promedio en esos datos de prueba. Ninguna métrica tiene sentido sin la escala y el costo del negocio.

**Checkpoint:** la regresión simple con IMC debería mejorar poco. Eso no significa que la regresión “no sirva”; significa que un único predictor no contiene suficiente información.


En la regresión múltiple, las variables numéricas se estandarizan y las categorías se convierten en indicadores. Todo se aprende dentro del pipeline usando solo entrenamiento. La categoría `smoker` ayuda a mostrar por qué una relación bivariada débil puede cambiar al incorporar información pertinente.


In [ ]:
reg_num = ["age", "bmi", "children"]
reg_cat = ["sex", "smoker", "region"]

prep_reg = ColumnTransformer([
    ("numericas", StandardScaler(), reg_num),
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), reg_cat),
])

multiple = Pipeline([
    ("preparar", prep_reg),
    ("modelo", LinearRegression()),
])

multiple.fit(Xr_train, yr_train)
pred_multiple = multiple.predict(Xr_test)
resultados_reg.append(metricas_regresion("Lineal múltiple", yr_test, pred_multiple))

tabla_reg = pd.DataFrame(resultados_reg).set_index("modelo")
display(tabla_reg.round(2))


### Paso 4 · ¿Qué aprendió la regresión múltiple?

Las categorías se transformaron en columnas 0/1 mediante **one-hot encoding**. La estandarización pone las variables numéricas en escalas comparables para el proceso de modelado. El pipeline garantiza que las transformaciones se aprendan solo con entrenamiento.

Los coeficientes muestran dirección dentro del modelo. Como los predictores numéricos fueron estandarizados, no deben leerse directamente como “pesos por un año” sin deshacer la escala.


In [ ]:
nombres_reg = multiple.named_steps["preparar"].get_feature_names_out()
coeficientes_reg = pd.Series(
    multiple.named_steps["modelo"].coef_,
    index=nombres_reg,
    name="coeficiente"
).sort_values()

display(pd.concat([
    coeficientes_reg.head(6),
    coeficientes_reg.tail(6)
]).to_frame().round(2))

coeficientes_reg.sort_values().plot(
    kind="barh", figsize=(8, 6), color="#0E5A9C"
)
plt.title("Dirección y magnitud interna de coeficientes")
plt.xlabel("Coeficiente en el espacio transformado")
plt.tight_layout()
plt.show()


In [ ]:
comparacion = pd.DataFrame({
    "real": yr_test,
    "prediccion": pred_multiple,
})
comparacion["error_absoluto"] = (comparacion["real"] - comparacion["prediccion"]).abs()
display(comparacion.sort_values("error_absoluto", ascending=False).head(10).round(0))

plt.figure(figsize=(6, 5))
sns.scatterplot(data=comparacion, x="real", y="prediccion", alpha=.7)
limite = max(comparacion["real"].max(), comparacion["prediccion"].max())
plt.plot([0, limite], [0, limite], "--", color="firebrick", label="predicción perfecta")
plt.title("Valor real vs. predicción fuera de muestra")
plt.legend()
plt.tight_layout()
plt.show()


### Paso 5 · Auditar los residuos

Un residuo es `real − predicción`:

- positivo: el modelo subestimó;
- negativo: el modelo sobreestimó;
- cercano a cero: predicción próxima al observado.

La tabla anterior muestra los casos más costosos. Pregunta si comparten región, condición de fumador, rangos de IMC u otra característica. Los errores agregados pueden ocultar un segmento donde el modelo falla sistemáticamente.


In [ ]:
auditoria_reg = Xr_test.copy()
auditoria_reg["real"] = yr_test
auditoria_reg["prediccion"] = pred_multiple
auditoria_reg["residuo"] = auditoria_reg["real"] - auditoria_reg["prediccion"]
auditoria_reg["error_absoluto"] = auditoria_reg["residuo"].abs()

error_segmento = (
    auditoria_reg.groupby("smoker")
    .agg(
        casos=("error_absoluto", "size"),
        MAE=("error_absoluto", "mean"),
        sesgo_medio=("residuo", "mean"),
    )
)
display(error_segmento.round(2))


### Escenario de sensibilidad — no causal

La siguiente celda usa el modelo para estimar dos perfiles. Cambiar una variable y observar otra predicción muestra **sensibilidad del modelo**, no el efecto causal de intervenir sobre una persona.


In [ ]:
escenarios = pd.DataFrame([
    {"age": 40, "bmi": 30, "children": 1,
     "sex": "female", "smoker": "no", "region": "northeast"},
    {"age": 40, "bmi": 30, "children": 1,
     "sex": "female", "smoker": "yes", "region": "northeast"},
])
escenarios["costo_estimado"] = multiple.predict(escenarios)
display(escenarios.round(2))


**Interpretación obligatoria**

- ¿Cuál modelo supera la línea base?
- Expresa el MAE en las unidades monetarias de `charges`: ¿es tolerable para qué decisión?
- ¿Los errores grandes se concentran en pólizas extraordinarias?
- Un buen R² **no** demuestra que fumar, edad o IMC causen el costo observado.

### Mini-reto · 15 minutos

Retira `smoker` de `variables_reg`, vuelve a ejecutar desde la partición y compara MAE/R². Explica:

1. cuánto cambia la capacidad predictiva;
2. por qué importancia predictiva no es causalidad;
3. qué riesgo tendría usar este modelo para fijar una tarifa real.


## Laboratorio 3 · Clasificación de cancelación

**Pregunta:** ¿podemos ordenar transacciones según su probabilidad de cancelarse?

### De probabilidad a decisión

Un clasificador no necesariamente empieza diciendo “sí/no”. Primero puede producir una probabilidad, por ejemplo `0,37`. Después usamos un umbral:

```text
si probabilidad ≥ umbral  →  generar alerta
si probabilidad < umbral  →  no generar alerta
```

Bajar el umbral produce más alertas y normalmente mayor recall, pero también más falsos positivos. El umbral es una decisión de negocio, no una constante matemática.

Este es un caso desbalanceado: la mayoría de ventas se completa. Compararemos una línea base, logística, árbol y Random Forest. Si los modelos no encuentran señal útil, esa también es una conclusión válida.

### Familias que compararemos

| Modelo | Intuición | Fortaleza | Riesgo |
|---|---|---|---|
| Logística | combina variables para estimar probabilidad | referencia interpretable | relaciones demasiado simples |
| Árbol | divide datos mediante reglas sucesivas | visual y no lineal | inestabilidad/sobreajuste |
| Random Forest | promedia muchos árboles diversos | estabilidad y flexibilidad | menor transparencia |


In [ ]:
modelo["cancelada"] = (modelo["estado"] == "Cancelada").astype(int)
print(modelo["cancelada"].value_counts().rename(index={0: "Completada", 1: "Cancelada"}))
print("Tasa de cancelación:", f"{modelo['cancelada'].mean():.1%}")

variables_clf = [
    "cantidad", "descuento", "valor_venta", "precio_lista", "edad",
    "canal", "categoria", "region", "segmento"
]
X_clf = modelo[variables_clf]
y_clf = modelo["cancelada"]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_clf, y_clf, test_size=.25, random_state=SEMILLA, stratify=y_clf
)


### Desbalance y la trampa de accuracy

Si alrededor de 91 % de las transacciones se completa, una regla que siempre diga “Completada” obtiene aproximadamente 91 % de accuracy y **cero** detecciones de cancelación.

Por eso necesitamos observar:

| Métrica | Pregunta práctica |
|---|---|
| Accuracy | ¿qué porcentaje total fue correcto? |
| Precision | de las alertas, ¿qué proporción canceló? |
| Recall | de todas las cancelaciones, ¿cuántas detectamos? |
| F1 | ¿qué equilibrio hay entre precision y recall? |
| ROC-AUC | ¿el modelo ordena cancelaciones por encima de completadas? |

ROC-AUC ≈ 0,50 representa ordenamiento similar al azar; 1,00 sería separación perfecta. Una métrica aceptable depende del uso, costo y estabilidad.


In [ ]:
clf_num = ["cantidad", "descuento", "valor_venta", "precio_lista", "edad"]
clf_cat = ["canal", "categoria", "region", "segmento"]

def nuevo_preprocesador():
    return ColumnTransformer([
        ("numericas", StandardScaler(), clf_num),
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), clf_cat),
    ])

modelos_clf = {
    "Línea base": Pipeline([
        ("preparar", nuevo_preprocesador()),
        ("modelo", DummyClassifier(strategy="prior")),
    ]),
    "Logística": Pipeline([
        ("preparar", nuevo_preprocesador()),
        ("modelo", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEMILLA)),
    ]),
    "Árbol": Pipeline([
        ("preparar", nuevo_preprocesador()),
        ("modelo", DecisionTreeClassifier(max_depth=4, min_samples_leaf=12,
                                           class_weight="balanced", random_state=SEMILLA)),
    ]),
    "Random Forest": Pipeline([
        ("preparar", nuevo_preprocesador()),
        ("modelo", RandomForestClassifier(n_estimators=300, max_depth=6,
                                           min_samples_leaf=5, class_weight="balanced",
                                           random_state=SEMILLA, n_jobs=-1)),
    ]),
}

filas = []
predicciones = {}
probabilidades = {}

for nombre, estimador in modelos_clf.items():
    estimador.fit(Xc_train, yc_train)
    pred = estimador.predict(Xc_test)
    prob = estimador.predict_proba(Xc_test)[:, 1]
    predicciones[nombre] = pred
    probabilidades[nombre] = prob
    filas.append({
        "modelo": nombre,
        "accuracy": accuracy_score(yc_test, pred),
        "precision": precision_score(yc_test, pred, zero_division=0),
        "recall": recall_score(yc_test, pred, zero_division=0),
        "F1": f1_score(yc_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(yc_test, prob),
    })

tabla_clf = pd.DataFrame(filas).set_index("modelo")
display(tabla_clf.round(3))


### Lectura guiada de la tabla

1. La línea base domina en accuracy porque la clase mayoritaria es grande.
2. Revisa recall y ROC-AUC antes de decir que un modelo mejora.
3. Si ROC-AUC está apenas por encima de 0,50, los datos contienen poca capacidad de ordenamiento.
4. `class_weight="balanced"` obliga al modelo a prestar más atención a la clase minoritaria; no crea información que no existe.

**Conclusión metodológica posible:** “Con estas variables, no existe evidencia suficiente para automatizar una intervención”. Esa es una respuesta analítica válida.


### Matrices de confusión y costo del error

Lee cada matriz así:

- falso positivo: intervengo una venta que no se cancelaría;
- falso negativo: no intervengo una venta que sí se cancela.

Accuracy no basta: la línea base puede ser muy “precisa” solo por repetir la clase mayoritaria.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, nombre in zip(axes, ["Logística", "Árbol", "Random Forest"]):
    ConfusionMatrixDisplay.from_predictions(
        yc_test, predicciones[nombre],
        display_labels=["Completa", "Cancela"],
        cmap="Blues", colorbar=False, ax=ax
    )
    ax.set_title(nombre)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for nombre in ["Logística", "Árbol", "Random Forest"]:
    RocCurveDisplay.from_predictions(
        yc_test, probabilidades[nombre], name=nombre, ax=ax
    )
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.set_title("Capacidad de ordenar riesgo en distintos umbrales")
plt.tight_layout()
plt.show()


### Cambiar el umbral

Si el falso negativo cuesta más, podemos bajar el umbral y detectar más cancelaciones, a cambio de más falsas alarmas. Ejecuta con 0,30 y después prueba 0,20 y 0,50.


In [ ]:
UMBRAL = 0.30
prob_rf = probabilidades["Random Forest"]
pred_umbral = (prob_rf >= UMBRAL).astype(int)

pd.Series({
    "umbral": UMBRAL,
    "precision": precision_score(yc_test, pred_umbral, zero_division=0),
    "recall": recall_score(yc_test, pred_umbral, zero_division=0),
    "F1": f1_score(yc_test, pred_umbral, zero_division=0),
    "alertas": int(pred_umbral.sum()),
})


### Comparar varios umbrales, no uno solo

La tabla siguiente hace visible el intercambio. En un proceso real también agregaríamos costo por alerta, beneficio por cancelación evitada y capacidad máxima del equipo.


In [ ]:
comparacion_umbrales = []
for umbral in [0.20, 0.30, 0.40, 0.50, 0.60]:
    pred_u = (prob_rf >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(yc_test, pred_u).ravel()
    comparacion_umbrales.append({
        "umbral": umbral,
        "alertas": int(pred_u.sum()),
        "precision": precision_score(yc_test, pred_u, zero_division=0),
        "recall": recall_score(yc_test, pred_u, zero_division=0),
        "falsos_positivos": int(fp),
        "falsos_negativos": int(fn),
    })

comparacion_umbrales = pd.DataFrame(comparacion_umbrales)
display(comparacion_umbrales.round(3))


**Pregunta de decisión:** si el equipo solo puede revisar 20 alertas, ¿qué umbral sería operativamente viable? ¿La baja precision justificaría contactar clientes?

No elijas el umbral que “se ve mejor” sin definir antes el costo de FP, FN y la capacidad disponible.


**Discusión obligatoria**

1. ¿Qué modelo supera realmente la línea base y en qué métrica?
2. ¿Hay evidencia suficiente para desplegar una intervención?
3. ¿Qué información adicional podría ser predictiva de cancelación?
4. ¿Qué costo operativo tendría bajar el umbral?

Un resultado cercano al azar no es un fracaso del notebook: muestra que **tener un algoritmo no garantiza tener señal**.

### Auditoría por segmento

Un promedio de desempeño puede esconder que el modelo falla más en un canal o región. La siguiente celda usa el umbral 0,30 para comparar tasas de error por canal.


In [ ]:
auditoria_clf = Xc_test[["canal", "region"]].copy()
auditoria_clf["real"] = yc_test.values
auditoria_clf["pred"] = (prob_rf >= .30).astype(int)
auditoria_clf["acierto"] = auditoria_clf["real"] == auditoria_clf["pred"]
auditoria_clf["falso_negativo"] = ((auditoria_clf["real"] == 1) &
                                    (auditoria_clf["pred"] == 0))

por_canal = (
    auditoria_clf.groupby("canal")
    .agg(
        casos=("real", "size"),
        cancelaciones=("real", "sum"),
        accuracy=("acierto", "mean"),
        falsos_negativos=("falso_negativo", "sum"),
    )
)
display(por_canal.round(3))


## Laboratorio 4 · Ver un árbol y comparar con el bosque

### Cómo aprende un árbol

Un árbol busca cortes que separen observaciones en grupos cada vez más homogéneos:

```text
nodo inicial
   ├── cumple una regla  → nueva división
   └── no la cumple      → nueva división
```

- **raíz:** contiene todas las observaciones;
- **nodo:** punto donde se evalúa una regla;
- **rama:** resultado de la regla;
- **hoja:** predicción final del recorrido;
- **profundidad:** número aproximado de decisiones sucesivas.

Un árbol profundo puede ajustarse a excepciones del entrenamiento y generalizar mal. `max_depth` y `min_samples_leaf` controlan esa flexibilidad.

El árbol es fácil de explicar; el bosque suele ser más estable. Visualizamos el árbol ajustado y revisamos importancia de variables del Random Forest. Ninguna importancia debe leerse como causalidad.


In [ ]:
arbol_pipe = modelos_clf["Árbol"]
nombres_arbol = arbol_pipe.named_steps["preparar"].get_feature_names_out()
arbol = arbol_pipe.named_steps["modelo"]

plt.figure(figsize=(20, 9))
plot_tree(
    arbol,
    feature_names=nombres_arbol,
    class_names=["Completa", "Cancela"],
    filled=True, rounded=True, proportion=True, fontsize=8
)
plt.title("Árbol de decisión (profundidad máxima = 4)")
plt.show()


In [ ]:
# Las mismas reglas en formato textual facilitan la lectura durante clase.
print(export_text(arbol, feature_names=list(nombres_arbol), max_depth=3))


### ¿Por qué un bosque puede mejorar?

Random Forest construye muchos árboles con muestras y subconjuntos de variables distintos. Después agrega sus resultados.

```text
árbol 1 ─┐
árbol 2 ─┼──► voto/promedio ─► predicción del bosque
árbol 3 ─┘
```

La diversidad reduce la dependencia de una sola estructura inestable. El costo es que ya no podemos resumir toda la decisión en un único diagrama.


In [ ]:
rf_pipe = modelos_clf["Random Forest"]
nombres_rf = rf_pipe.named_steps["preparar"].get_feature_names_out()
importancias = pd.Series(
    rf_pipe.named_steps["modelo"].feature_importances_,
    index=nombres_rf
).sort_values(ascending=False).head(12)

display(importancias.to_frame("importancia").round(3))
importancias.sort_values().plot(kind="barh", figsize=(7, 5), color="#0E5A9C")
plt.title("Variables más utilizadas por el Random Forest")
plt.xlabel("Importancia interna del modelo")
plt.tight_layout()
plt.show()


### Importancia no significa efecto

Una variable puede recibir importancia porque ayuda a dividir los datos, porque está correlacionada con otra o porque tiene muchos posibles puntos de corte. La gráfica anterior no permite decir:

> “Si cambiamos esta variable, causaremos una reducción de cancelaciones”.

Solo permite decir que el bosque la utilizó para construir sus predicciones en estos datos.

### Microactividad · 15 minutos

Cambia `max_depth=4` del árbol por `2` y después por `8`. Reentrena y compara:

- facilidad de explicación;
- precision/recall de prueba;
- diferencia entre desempeño de entrenamiento y prueba;
- cantidad de hojas.

Formula con tus palabras el intercambio **interpretabilidad ↔ flexibilidad ↔ sobreajuste**.


## Laboratorio 5 · Segmentación de clientes con K-means

**Pregunta:** ¿aparecen perfiles de comportamiento de compra sin proporcionar etiquetas previas?

### Qué es clustering

Clustering agrupa observaciones buscando alta similitud dentro de cada grupo y diferencias entre grupos. No recibe una columna con “segmento correcto”. Por eso el resultado es una propuesta de estructura, no una verdad revelada.

### Intuición de K-means

1. Elegir `k` centros iniciales.
2. Asignar cada observación al centro más cercano.
3. Recalcular cada centro como el promedio de su grupo.
4. Repetir hasta que las asignaciones se estabilicen.

La “cercanía” se calcula usando las variables que nosotros escogemos. Cambiar variables, escala o `k` puede cambiar por completo la segmentación.

Primero cambiamos la unidad de análisis: una fila dejará de ser transacción y pasará a ser cliente. Después estandarizamos las métricas.

### Variables construidas

| Métrica | Qué representa | Riesgo interpretativo |
|---|---|---|
| transacciones | frecuencia observada | depende del periodo |
| gasto total | aporte agregado | dominado por escala |
| ticket promedio | tamaño medio de operación | sensible a extremos |
| descuento promedio | exposición/uso de descuento | no mide sensibilidad causal |
| tasa de cancelación | proporción cancelada | inestable con pocas compras |


In [ ]:
clientes_kpi = (
    modelo
    .groupby("id_cliente", as_index=False)
    .agg(
        transacciones=("id_venta", "nunique"),
        compras_completadas=("cancelada", lambda s: int((s == 0).sum())),
        gasto_total=("valor_venta", lambda s: s[modelo.loc[s.index, "cancelada"] == 0].sum()),
        ticket_promedio=("valor_venta", "mean"),
        descuento_promedio=("descuento", "mean"),
        tasa_cancelacion=("cancelada", "mean"),
    )
)

variables_cluster = [
    "transacciones", "gasto_total", "ticket_promedio",
    "descuento_promedio", "tasa_cancelacion"
]
scaler_cluster = StandardScaler()
Z = scaler_cluster.fit_transform(clientes_kpi[variables_cluster])

print("Unidad de análisis: cliente")
print("Clientes:", len(clientes_kpi))
display(clientes_kpi.head())


### Por qué estandarizamos

`gasto_total` puede estar en cientos de miles mientras `tasa_cancelacion` está entre 0 y 1. Sin estandarizar, la distancia quedaría dominada por la variable de mayor escala.

La estandarización transforma cada valor aproximadamente así:

```text
valor estandarizado = (valor − media) / desviación estándar
```

- `0`: cerca de la media;
- `1`: una desviación por encima;
- `−1`: una desviación por debajo.

No elimina outliers ni vuelve buenas las variables; solo las lleva a una escala comparable.


In [ ]:
ejemplo_escala = pd.DataFrame({
    "variable": variables_cluster,
    "min_original": clientes_kpi[variables_cluster].min(),
    "max_original": clientes_kpi[variables_cluster].max(),
    "media_estandarizada": Z.mean(axis=0),
    "sd_estandarizada": Z.std(axis=0),
})
display(ejemplo_escala.round(3))


In [ ]:
evaluacion_k = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=30, random_state=SEMILLA)
    etiquetas = km.fit_predict(Z)
    evaluacion_k.append({
        "k": k,
        "inercia": km.inertia_,
        "silhouette": silhouette_score(Z, etiquetas),
    })

evaluacion_k = pd.DataFrame(evaluacion_k)
display(evaluacion_k.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.lineplot(data=evaluacion_k, x="k", y="inercia", marker="o", ax=axes[0])
axes[0].set_title("Método del codo")
sns.lineplot(data=evaluacion_k, x="k", y="silhouette", marker="o", ax=axes[1])
axes[1].set_title("Silhouette: mayor suele ser mejor")
plt.tight_layout()
plt.show()


### Elegir `k`: criterio, no automatismo

- **Inercia:** suma de distancias internas; siempre baja al aumentar `k`, por eso buscamos un “codo”.
- **Silhouette:** compara cercanía al propio grupo frente al grupo vecino; va aproximadamente de −1 a 1.
- **Tamaño:** clusters diminutos pueden no ser accionables.
- **Estabilidad:** el perfil debería resistir cambios razonables de semilla/muestra.
- **Utilidad:** debe existir una acción diferenciada legítima.

El mejor silhouette no obliga a escoger esa solución. La decisión combina evidencia estadística y sentido empresarial.


Para la demostración elegimos `k = 3`. No es una verdad descubierta: es una decisión que debe contrastarse con estabilidad, tamaño y utilidad de los grupos.


In [ ]:
K = 3
kmeans = KMeans(n_clusters=K, n_init=50, random_state=SEMILLA)
clientes_kpi["cluster"] = kmeans.fit_predict(Z)

perfil_cluster = (
    clientes_kpi
    .groupby("cluster")[variables_cluster]
    .agg(["count", "mean", "median"])
)
display(perfil_cluster.round(2))

tamanos = clientes_kpi["cluster"].value_counts().sort_index()
display(tamanos.rename("clientes").to_frame())


In [ ]:
# Perfil relativo: 0 es la media global; positivo está por encima.
perfil_medio = clientes_kpi.groupby("cluster")[variables_cluster].mean()
perfil_relativo = (
    (perfil_medio - clientes_kpi[variables_cluster].mean()) /
    clientes_kpi[variables_cluster].std()
)

plt.figure(figsize=(9, 4))
sns.heatmap(perfil_relativo, annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Perfil relativo de cada cluster")
plt.xlabel("Métrica")
plt.ylabel("Cluster")
plt.tight_layout()
plt.show()


**Interpretación**

- El algoritmo devuelve números, no nombres empresariales.
- Propón un nombre solo después de revisar medias, medianas, tamaño y casos representativos.
- Un nombre como “Premium” debe ser una interpretación documentada, no una supuesta verdad del modelo.
- Antes de actuar, revisa estabilidad y posibles efectos injustos de segmentar personas.

### Nombrar con evidencia

Evita nombres valorativos como “buenos”, “malos” o “problemáticos”. Prefiere descripciones observables, por ejemplo:

- “mayor frecuencia y gasto total”;
- “baja frecuencia y ticket alto”;
- “mayor descuento y cancelación”.

Después revisa casos concretos para confirmar que el promedio no oculta heterogeneidad.


In [ ]:
for cluster_id in sorted(clientes_kpi["cluster"].unique()):
    print(f"\nCluster {cluster_id}: ejemplos cercanos al perfil medio")
    centro = kmeans.cluster_centers_[cluster_id]
    indices = np.where(clientes_kpi["cluster"].to_numpy() == cluster_id)[0]
    distancias = np.linalg.norm(Z[indices] - centro, axis=1)
    elegidos = indices[np.argsort(distancias)[:3]]
    display(clientes_kpi.iloc[elegidos][["id_cliente"] + variables_cluster])


### Mini-reto · 20 minutos

Compara `k=3` y `k=4`:

1. silhouette y tamaños;
2. perfiles relativos;
3. nombres descriptivos;
4. una acción posible por grupo;
5. un riesgo de uso.

Puedes concluir que ninguna solución es suficientemente estable o accionable. Justificar “no segmentar” también demuestra criterio.


## Profundización 1 · PCA para visualizar los perfiles

### Problema que resuelve

Con muchas variables no podemos visualizar directamente todas las dimensiones. PCA construye componentes como combinaciones ponderadas de las variables originales:

```text
PC1 = peso₁×transacciones + peso₂×gasto + ...
```

- **componente:** nueva dimensión resumida;
- **carga:** peso y dirección de cada variable en un componente;
- **score:** posición de un cliente sobre el componente;
- **varianza explicada:** información relativa conservada.

PCA resume las cinco métricas estandarizadas. La visualización en dos dimensiones ayuda a explorar separación, pero no prueba que los clusters sean “reales”. El componente maximiza variación matemática, no utilidad empresarial.


In [ ]:
pca = PCA(n_components=2)
componentes = pca.fit_transform(Z)
clientes_kpi["PC1"] = componentes[:, 0]
clientes_kpi["PC2"] = componentes[:, 1]

print("Varianza explicada por PC1 y PC2:",
      np.round(pca.explained_variance_ratio_, 3),
      "| total:", round(pca.explained_variance_ratio_.sum(), 3))

cargas = pd.DataFrame(
    pca.components_.T,
    index=variables_cluster,
    columns=["PC1", "PC2"]
)
display(cargas.round(3))

plt.figure(figsize=(7, 5))
sns.scatterplot(data=clientes_kpi, x="PC1", y="PC2",
                hue="cluster", palette="Set2", s=70, alpha=.8)
plt.title("Clientes proyectados en dos componentes")
plt.tight_layout()
plt.show()


**Pregunta:** ¿qué variables parecen definir PC1 y PC2 según sus cargas? Evita nombrar un componente a partir de una sola carga o sin revisar signos y magnitudes.

Si PC1 y PC2 explican cerca de 62 % de la variación, la gráfica conserva una parte importante pero pierde aproximadamente 38 %. Dos puntos cercanos en la gráfica pueden diferir en componentes no mostrados.


## Profundización 2 · Detección de anomalías

### Anomalía no significa error ni fraude

Una anomalía es una observación que se comporta de forma inusual según las variables y el método seleccionados. Puede ser:

- error de captura;
- evento válido y extraordinario;
- subgrupo no reconocido;
- caso prioritario para revisar.

Isolation Forest crea particiones aleatorias. Las observaciones fáciles de aislar reciben mayor puntuación de anomalía. Busca combinaciones inusuales de cantidad, descuento, valor y precio.

La proporción `contamination=.03` es una decisión: le pedimos marcar aproximadamente 3 % como anomalías para revisión. No es una tasa real de fraude.


In [ ]:
variables_anomalia = ["cantidad", "descuento", "valor_venta", "precio_lista"]
X_anomalia = modelo[variables_anomalia].copy()

iso = IsolationForest(contamination=.03, random_state=SEMILLA)
modelo["marca_anomalia"] = iso.fit_predict(X_anomalia)
modelo["score_anomalia"] = -iso.score_samples(X_anomalia)

casos_inusuales = (
    modelo.loc[modelo["marca_anomalia"] == -1,
               ["id_venta", "canal", "categoria", "estado"] + variables_anomalia + ["score_anomalia"]]
    .sort_values("score_anomalia", ascending=False)
)
print("Casos marcados:", len(casos_inusuales))
display(casos_inusuales.head(12))


Una anomalía **no** es automáticamente fraude ni error. Requiere contexto, verificación y un proceso de investigación. Cambia `contamination` a `.01` y `.05`: ¿cómo cambia la carga de revisión?

### Comparación con IQR

IQR examina una variable cada vez y es transparente. Isolation Forest puede detectar combinaciones multivariables inusuales, pero es menos intuitivo. En una organización puede ser útil comenzar con reglas simples y justificar la complejidad adicional.


## 8. Quick Reference final

| Quiero... | Modelo inicial | Qué debo evaluar |
|---|---|---|
| estimar un número | regresión + línea base | MAE/RMSE fuera de muestra |
| predecir una clase | logística/árbol/RF | precision, recall, umbral y costo |
| encontrar perfiles | K-means | escala, `k`, silhouette, estabilidad y utilidad |
| resumir variables | PCA | varianza explicada y cargas |
| investigar excepciones | IQR/Isolation Forest | falsos positivos y revisión humana |

### Regla de salida

> Un modelo útil no es el que tiene el nombre más avanzado. Es el que supera una línea base, generaliza, se interpreta con límites y mejora una decisión a un costo aceptable.


## 8.1 Árbol de decisión conceptual para escoger técnica

```text
¿Tengo una respuesta histórica conocida?
│
├── Sí
│   ├── Es numérica      → regresión
│   └── Es categoría     → clasificación
│
└── No
    ├── Busco grupos     → clustering
    ├── Resumir variables→ PCA
    └── Casos inusuales  → anomalías
```

Después pregunta:

1. ¿la unidad de análisis es correcta?;
2. ¿los predictores existen en el momento de decidir?;
3. ¿cuál es la línea base?;
4. ¿qué error cuesta más?;
5. ¿quién actuará y qué capacidad tiene?;
6. ¿qué sesgo o daño podría producirse?;
7. ¿qué evidencia haría que descartemos el modelo?


## 8.2 Errores frecuentes al trabajar con IA

| Error | Por qué importa | Control |
|---|---|---|
| inventar columnas | el código no corresponde a los datos | entregar esquema real |
| preparar antes de dividir | fuga entre entrenamiento y prueba | pipeline |
| evaluar sobre entrenamiento | optimismo artificial | conjunto de prueba |
| escoger por accuracy | ignora desbalance/costos | matriz + precision/recall |
| llamar “premium” a un cluster | etiqueta subjetiva no validada | perfil + casos + acción |
| interpretar importancia como causa | excede la evidencia | lenguaje asociativo |
| copiar sin ejecutar | no hay reproducibilidad | reiniciar y ejecutar todo |

Prompt de auditoría:

```text
Revisa este pipeline como auditor. Identifica unidad de análisis, X, y,
momento de disponibilidad de cada predictor, separación train/test,
línea base, métrica, desbalance, posible fuga y afirmaciones causales
no respaldadas. No propongas código hasta enumerar los riesgos.
```


## 9. Mini-reto evaluable

Elige **uno**:

**A. Clasificación**  
Modifica el umbral de Random Forest. Construye una tabla con precision, recall, número de alertas, falsos positivos y falsos negativos para tres umbrales. Recomienda uno o concluye que no hay evidencia para usar el modelo.

**B. Segmentación**  
Compara `k = 2`, `3` y `4`. Presenta silhouette, tamaños, perfil y estabilidad interpretativa. Recomienda una solución o concluye que los grupos no son accionables.

**Entregable:** una tabla, una visualización, dos hallazgos, una decisión y una limitación.

### Preguntas para la socialización

- ¿Qué esperabas antes de ejecutar?
- ¿Qué resultado contradijo tu expectativa?
- ¿Qué decisión cambiaría con la salida?
- ¿Qué resultado adicional necesitarías?
- ¿En qué situación no usarías este modelo?


## 10. Transferencia al proyecto de grado

Completa una ficha:

```text
Pregunta y decisión:
Unidad de análisis:
Tipo de tarea:
Variable objetivo (si existe):
Datos disponibles al decidir:
Línea base:
Métrica:
Error más costoso:
Riesgo de fuga o sesgo:
Evidencia mínima para recomendar uso:
```

No es obligatorio aplicar un modelo complejo. También es una decisión analítica válida documentar que los datos actuales solo permiten descripción o exploración.
